# Teste local → Git → AIDP → Autonomous Lakehouse
Adaptado de [Access_ALH_Data.ipynb](https://github.com/oracle-samples/oracle-aidp-samples/blob/main/getting-started/Access_ALH_Data.ipynb).

Oracle AI Data Platform v1.0 — Copyright © 2025, Oracle and/or its affiliates.
Licensed under the Universal Permissive License v1.0: https://oss.oracle.com/licenses/upl/

Edite no VS Code e execute no compute do AIDP. O kernel Python local não fornece `spark` nem `oidlUtils`.
O catálogo `oracle_26ai_lakehouse` já contém a conexão: este notebook não lê `.env` ou wallet.
Por padrão, descobre os schemas. Informe SCHEMA e TABLE para testar leitura de uma tabela.
A escrita é opcional, usa dados sintéticos e exige uma tabela de teste existente com o schema do exemplo original.

In [ ]:
# Parâmetros do workflow; valores padrão permitem um smoke test de descoberta.
import json
import re
from uuid import uuid4

if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute este notebook no compute AIDP; o VS Code é usado para edição local.')

def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, default)
    return default if value is None else str(value).strip()

def identifier(value):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        raise ValueError(f'Identificador inválido: {value!r}')
    return f'`{value}`'

catalog = parameter('CATALOG', 'oracle_26ai_lakehouse')
schema = parameter('SCHEMA', '')
table = parameter('TABLE', '')
write_enabled = parameter('WRITE_ENABLED', 'false').lower() == 'true'
if bool(schema) != bool(table):
    raise ValueError('Informe SCHEMA e TABLE juntos, ou deixe ambos vazios.')
if write_enabled and not table:
    raise ValueError('A escrita exige SCHEMA e TABLE de teste explícitos.')
run_id = uuid4().hex[:12]
result = {'run_id': run_id, 'catalog': catalog, 'write_enabled': write_enabled}

In [ ]:
# Smoke test: força acesso ao catálogo remoto e imprime somente nomes de schemas.
rows = spark.sql(f'SHOW SCHEMAS IN {identifier(catalog)}').collect()
result['schema_count'] = len(rows)
print(json.dumps({'schemas': [r.asDict() for r in rows]}, default=str))

In [ ]:
# Leitura opcional: força execução com uma projeção limitada; não imprime dados da tabela.
target = None
if schema and table:
    target = '.'.join(identifier(x) for x in (catalog, schema, table))
    remote = spark.read.table(target)
    result['table'] = f'{catalog}.{schema}.{table}'
    result['sample_rows_read'] = remote.limit(5).count()
    result['column_count'] = len(remote.columns)
else:
    result['table_read'] = 'SKIPPED: informe SCHEMA e TABLE para validar a leitura'

## Escrita opcional em tabela de teste
Defina `WRITE_ENABLED=true` somente com SCHEMA/TABLE de teste. Cada execução adiciona três registros identificados por um prefixo único; eles permanecem no banco após o teste.

Crie previamente uma tabela com as colunas do exemplo original: `patient_id VARCHAR2(30)`, `heart_rate_bpm NUMBER`, `blood_pressure VARCHAR2(30)`, `oxygen_saturation NUMBER`, `room_number VARCHAR2(30)`, `status VARCHAR2(30)`. Use tipos de coluna compatíveis. O notebook não cria nem altera tabelas.

In [ ]:
if write_enabled:
    from pyspark.sql import functions as F
    columns = ['patient_id', 'heart_rate_bpm', 'blood_pressure', 'oxygen_saturation', 'room_number', 'status']
    remote = spark.read.table(target)
    actual = {name.lower(): name for name in remote.columns}
    if set(actual) != set(columns) or len(remote.columns) != len(columns):
        raise ValueError('A tabela de teste deve ter exatamente as seis colunas documentadas.')
    prefix = f'T{run_id}'
    records = [
        (prefix + '_1', 72, '120/80', 98, 'TEST', 'synthetic'),
        (prefix + '_2', 95, '145/95', 90, 'TEST', 'synthetic'),
        (prefix + '_3', 88, '125/95', 92, 'TEST', 'synthetic'),
    ]
    source = spark.createDataFrame(records, columns)
    # insertInto usa posição: respeitar a ordem e os tipos da tabela remota.
    ordered = source.select(*[
        F.col(field.name.lower()).cast(field.dataType).alias(field.name)
        for field in remote.schema.fields
    ])
    ordered.filter(F.col(actual['patient_id']).isin([records[0][0], records[1][0]])).write.mode('append').insertInto(target)
    ordered.filter(F.col(actual['patient_id']) == records[2][0]).createOrReplaceTempView('alh_test_sql_source')
    spark.sql(f'INSERT INTO {target} SELECT * FROM alh_test_sql_source')
    ids = [r[0] for r in records]
    verified = spark.read.table(target).filter(F.col(actual['patient_id']).isin(ids)).count()
    if verified != 3:
        raise AssertionError(f'Esperados 3 registros de teste; encontrados {verified}.')
    result.update(rows_written=verified, test_patient_ids=ids)

In [ ]:
# Marcador de validação para localizar no output do workflow.
result['status'] = 'SUCCESS'
print('AIDP_ALH_TEST_RESULT=' + json.dumps(result, sort_keys=True))